In [6]:
import gc
import torch

del llm

# 3. Force Python garbage collection
gc.collect()

# 4. Clear CUDA cache (Required if running on GPU)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [2]:
# llma-cpp-python을 사용해 GGUF 파일을 로드한다.
from langchain_community.llms import LlamaCpp

llm  = LlamaCpp(
    model_path="D:\.lmstudio\models\lmstudio-community\gemma-4-26B-A4B-it-QAT-GGUF\gemma-4-26B-A4B-it-QAT-Q4_0.gguf",
    n_gpu_layers=-1, # n_gpu_layers: GPU에서 사용할 레이어 수를 -1로 설정하여 모든 레이어를 GPU에서 실행한다.
    max_tokens=500, # max_tokens: 최대 토큰 수를 500으로 설정한다.
    c_ctx_size=4096, # c_ctx_size: 컨텍스트 크기를 4096으로 설정한다.
    seed=42, # seed: 시드를 42로 설정하여 결과의 일관성을 유지한다.
    verbose=False, # verbose: 자세한 로그 출력을 비활성화한다.
    )

<>:5: SyntaxWarning: invalid escape sequence '\.'
<>:5: SyntaxWarning: invalid escape sequence '\.'
C:\Users\ParkJuYeong\AppData\Local\Temp\ipykernel_5396\3057028141.py:5: SyntaxWarning: invalid escape sequence '\.'
  model_path="D:\.lmstudio\models\lmstudio-community\gemma-4-26B-A4B-it-QAT-GGUF\gemma-4-26B-A4B-it-QAT-Q4_0.gguf",
c:\Users\ParkJuYeong\anaconda3\envs\conda_python313\Lib\site-packages\IPython\core\interactiveshell.py:3715: UserWarning: WARNING! c_ctx_size is not default parameter.
                c_ctx_size was transferred to model_kwargs.
                Please confirm that c_ctx_size is what you intended.
  if await self.run_code(code, result, async_=asy):
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA i

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import LLMChain

# 대화 기록을 담을 수 있도록 프롬프트를 업데이트합니다.
template = """<|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)
# 사용할 메모리를 정의합니다.
memory = ConversationBufferMemory(memory_key="chat_history")

# LLM, 프롬프트, 메모리를 연결합니다.
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\ParkJuYeong\AppData\Local\Temp\ipykernel_5396\1310473298.py:16: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")
C:\Users\ParkJuYeong\AppData\Local\Temp\ipykernel_5396\1310473298.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


In [5]:
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': 'Human: Hi! My name is Maarten. What is 1 + 1?\nAI: \n<|channel>|assistant|>Hi Maarten! Nice to meet you. 1 + 1 is 2. Is there anything else I can help you with?<|end|>',
 'text': 'Hi Maarten! Nice to meet you. 1 + 1 is 2. Is there anything else I can help you with?<|end|>\n<|channel>thought\n<channel|>Hi Maarten! Nice to meet you. 1 + 1 is 2. Is there anything else I can help you with?'}

In [6]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hi! My name is Maarten. What is 1 + 1?\nAI: \n<|channel>|assistant|>Hi Maarten! Nice to meet you. 1 + 1 is 2. Is there anything else I can help you with?<|end|>\nHuman: Hi! My name is Maarten. What is 1 + 1?\nAI: Hi Maarten! Nice to meet you. 1 + 1 is 2. Is there anything else I can help you with?<|end|>\n<|channel>thought\n<channel|>Hi Maarten! Nice to meet you. 1 + 1 is 2. Is there anything else I can help you with?',
 'text': "Your name is Maarten. Is there anything else you'd like to ask?"}